In [1]:
from google import genai
from google.genai.types import EmbedContentConfig
from google.genai import types
import pandas as pd
import time
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

In [2]:
client = genai.Client(vertexai=True, project="resume-radar-thuan", location="us-central1")

In [3]:
bq = bigquery.Client(project="resume-radar-thuan")

In [4]:
df = pd.read_csv('data/postings.csv')
df = df.dropna(subset = ['formatted_experience_level'])
df['text'] = df.title +  ' ' + df.description.fillna("")
X = df.text
y = df.formatted_experience_level
df['text'].str.len().mean()

np.float64(3910.1427996611606)

In [5]:
df_sample = df.sample(n=10000, random_state = 42)

In [6]:
try:
    done = {row.job_id for row in bq.query("SELECT job_id FROM resume_radar.posting_embeddings").result()}
except NotFound:
    done = set()

In [7]:
batch_size = 100

try:
    done = {row.job_id for row in bq.query("SELECT job_id FROM resume_radar.posting_embeddings").result()}
except NotFound:
    done = set()
todo = df_sample[~df_sample["job_id"].isin(done)]
for start in range(0, len(todo), batch_size):
    batch = todo.iloc[start:start +batch_size]
    
    for attempt in range(5):
        try:
            embed_content = client.models.embed_content(
                model="gemini-embedding-001",
                contents=batch.text.to_list(),
                config=types.EmbedContentConfig(output_dimensionality=768),)
            break
        except Exception as e:
            print(f"batch {start}: attempt {attempt} failed: {e}")
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError(f"batch {batch} failed 5 times")
    
    rows = pd.DataFrame({
    "job_id": batch["job_id"].to_list(),
    "embedding": [e.values for e in embed_content.embeddings],
    "model": "gemini-embedding-001-768",
    })
    bq.load_table_from_dataframe(rows, "resume_radar.posting_embeddings").result()
    print(f"done {start + len(batch)} / {len(todo)}")



In [8]:
query = """
WITH target AS (
  SELECT embedding FROM resume_radar.posting_embeddings WHERE job_id = 3905341299
)
SELECT
  p.title,
  p.job_id,
  ML.DISTANCE(target.embedding, o.embedding, 'COSINE') AS cos_dist
FROM resume_radar.posting_embeddings AS o
CROSS JOIN target
JOIN resume_radar.postings AS p ON p.job_id = o.job_id
WHERE o.job_id != 3905341299
ORDER BY cos_dist ASC      
LIMIT 10
"""
for row in bq.query(query).result():
    print(row)

Row(('Web Designer I', 3904402111, 0.21990431061940008), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist', 3904363560, 0.23739560201518262), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Senior Data Scientist, Experimentation & Personalization', 3894285749, 0.2495690208598913), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist I', 3904960773, 0.26596970265235476), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Analytics Consultant', 3900089227, 0.26903815818091936), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist Lead – Telematics (Remote)', 3894627680, 0.27370603422463313), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist', 3906226694, 0.2812813937142221), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Data Scientist III, Innovation', 3905885501, 0.2866950120680073), {'title': 0, 'job_id': 1, 'cos_dist': 2})
Row(('Associate Predictive Modeler II', 3885828791, 0.2899849299257812), {'title': 0, 'job_id': 1, 'cos_dist': 2}

In [9]:
query = """
SELECT job_id, title FROM resume_radar.postings 
WHERE job_id IN (SELECT job_id FROM resume_radar.posting_embeddings)
AND LOWER(title) LIKE '%data scientist%' LIMIT 5
"""
for row in bq.query(query).result():
    print(row)

Row((3905341299, 'Data Scientist'), {'job_id': 0, 'title': 1})
Row((3904960773, 'Data Scientist I'), {'job_id': 0, 'title': 1})
Row((3885812100, 'junior java programmer/data scientist'), {'job_id': 0, 'title': 1})
Row((3905885501, 'Data Scientist III, Innovation'), {'job_id': 0, 'title': 1})
Row((3902834352, 'Vice President/P&C Actuary/Data Scientist - PR12682'), {'job_id': 0, 'title': 1})


In [84]:
anchors = df_sample.sample(n=10, random_state= 7)[['job_id','title']]
anchors

,job_id,title
1894,3884440227,National Team Manager - Construction Material...
22323,3889711039,Construction Superintendent
96117,3904942511,Accounting Manager
101037,3905239556,Python Developer
95373,3904928589,"Industry Sales Executive- Communications, Medi..."
108109,3905336539,Families First Case Manager
33468,3895210127,Senior Information Technology Manager
96156,3904942873,Remote Client Services Rep (Insurance) - No Ex...
76391,3903471528,Combination Building Inspector
42768,3899542292,Consumer Loan Sales Specialist


In [87]:
sample = []
for job in anchors.itertuples():
    query = f"""
    WITH target AS (
      SELECT e.embedding, e.job_id, p2.title AS anchor_title, p2.company_name AS anchor_company 
      FROM resume_radar.posting_embeddings e 
      JOIN resume_radar.postings p2
      ON p2.job_id = e.job_id
      WHERE e.job_id = {job[1]}
    )
    SELECT
      p.title,
      target.anchor_title,
      target.anchor_company,
      target.job_id,
      p.job_id,
      p.company_name,
      ML.DISTANCE(target.embedding, o.embedding, 'COSINE') AS cos_dist
    FROM resume_radar.posting_embeddings AS o
    CROSS JOIN target
    JOIN resume_radar.postings AS p ON p.job_id = o.job_id
    WHERE o.job_id != {job[1]}
    ORDER BY cos_dist ASC      
    LIMIT 3
    """
    for row in bq.query(query).result():
        sample.append(dict(row))

[{'title': 'National Practice  Manager - Construction Materials Testing', 'anchor_title': 'National Team  Manager - Construction Materials Testing', 'anchor_company': 'Atlas', 'job_id': 3884440227, 'job_id_1': 3884439422, 'company_name': 'Atlas', 'cos_dist': 0.015809860535369813}, {'title': 'Resident Engineer', 'anchor_title': 'National Team  Manager - Construction Materials Testing', 'anchor_company': 'Atlas', 'job_id': 3884440227, 'job_id_1': 3902868071, 'company_name': 'Atlas', 'cos_dist': 0.21449277382558984}, {'title': 'Structural Bridge Engineering Manager #24-170', 'anchor_title': 'National Team  Manager - Construction Materials Testing', 'anchor_company': 'Atlas', 'job_id': 3884440227, 'job_id_1': 3886884354, 'company_name': 'Atlas', 'cos_dist': 0.2176706755484592}, {'title': 'Construction Superintendent', 'anchor_title': 'Construction Superintendent', 'anchor_company': 'Allegiance Technology', 'job_id': 3889711039, 'job_id_1': 3888973746, 'company_name': 'gpac', 'cos_dist': 0.

In [92]:
the_list = pd.DataFrame(sample)
the_list['relevant'] = 0

In [102]:
#the_list.to_csv("eval.csv")
the_list_evaluated = pd.read_csv("eval.csv")
the_list_evaluated

,Unnamed: 0,title,anchor_title,anchor_company,job_id,job_id_1,company_name,cos_dist,relevant
0,0,National Practice Manager - Construction Mate...,National Team Manager - Construction Material...,Atlas,3884440227,3884439422,Atlas,0.015810,0
1,1,Resident Engineer,National Team Manager - Construction Material...,Atlas,3884440227,3902868071,Atlas,0.214493,1
2,2,Structural Bridge Engineering Manager #24-170,National Team Manager - Construction Material...,Atlas,3884440227,3886884354,Atlas,0.217671,1
3,3,Construction Superintendent,Construction Superintendent,Allegiance Technology,3889711039,3888973746,gpac,0.213209,1
4,4,Construction Superintendent,Construction Superintendent,Allegiance Technology,3889711039,3900973979,RightPro Staffing,0.216332,1
5,5,Construction Superintendent,Construction Superintendent,Allegiance Technology,3889711039,3905248917,Torque Consulting,0.222436,1
6,6,Accounting Manager,Accounting Manager,"Comprehensive Logistics Co., Inc.",3904942511,3902835482,Ledgent,0.189538,1
7,7,Accounting Manager,Accounting Manager,"Comprehensive Logistics Co., Inc.",3904942511,3901803465,Broadview Federal Credit Union,0.202997,1
8,8,Accounting Manager,Accounting Manager,"Comprehensive Logistics Co., Inc.",3904942511,3891068832,CloudMasters,0.212760,1
9,9,Python Developer,Python Developer,Digitive,3905239556,3905341346,Radiansys Inc.,0.258169,1
